In [34]:
####document data structure
from langchain_core.documents import Document

In [35]:

from langchain_community.document_loaders.csv_loader import CSVLoader

loader = CSVLoader(
    file_path="../data/bhagavad_gita_verses.csv",
    source_column="chapter_verse",
    metadata_columns=["chapter_title"],                          # only title goes to metadata
    content_columns=["chapter_number", "chapter_verse", "translation"],  # these three become page_content
)

documents = loader.load()
#documents
print(f"Total documents: {len(documents)}")
#print(documents[3].page_content)
#print("---")
#print(data[0].metadata)

##Text Splitter




Total documents: 640


In [36]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [37]:
#Text Splitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ".", " ",""],  # order matters
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    length_function=len,
)

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} Documents into {len(split_docs)} chunks")

#Example of one chunk
    if split_docs:
        print("\nExample Chunk: ")
        print(split_docs[0].page_content)
        print("-" * 50)
        print(split_docs[0].metadata)
    return split_docs

In [38]:
chunks = split_documents(documents)
#chunks


# Generate Embeddings for each chunk


Split 640 Documents into 640 chunks

Example Chunk: 
chapter_number: Chapter 1
chapter_verse: 1.1
translation: Dhritarashtra said: O Sanjay, after gathering on the holy field of Kurukshetra, and desiring to fight, what did my sons and the sons of Pandu do?
--------------------------------------------------
{'source': '1.1', 'row': 0, 'chapter_title': 'Arjun Viṣhād Yog'}


In [39]:
from dotenv import load_dotenv
import os
import numpy as np
import chromadb
from chromadb.config import Settings
import uuid
import hashlib
from typing import List, Dict, Any, Tuple

from langchain_google_genai import GoogleGenerativeAIEmbeddings



load_dotenv("../dev.properties")
gemini_key = os.getenv("GEMINI_API_KEY")
#print(gemini_key)

class EmbeddingManager:
    def __init__(self, api_key: str = gemini_key, model_name: str = "gemini-embedding-001"):
        self.model_name = model_name
        self.api_key = api_key
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading the Embedding Model: {self.model_name}")
            self.model = GoogleGenerativeAIEmbeddings(
                model=self.model_name,
                google_api_key=self.api_key
            )
            print(f"Model Loaded Successfully! ")
        except Exception as e:
            print(f"Error loading model: {e}")
            raise e
    def generateEmbeddings(self, text: str) -> np.ndarray:
        """Generate embeddings for a list of text
        Args:
            text (str): Text to generate embeddings for
        Returns:
            np.ndarray: Embeddings for the given text as numpy array with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded. Please load the model first.")
        print(f"Generate embeddings for {len(text)} documents....")
        embeddings = self.model.embed_documents(text)
        embeddings = np.array(embeddings)
        print(f"Embeddings generated successfully! Shape: {embeddings.shape}")
        return embeddings
        

# Usage
embedding_manager = EmbeddingManager()




Loading the Embedding Model: gemini-embedding-001
Model Loaded Successfully! 


In [28]:
'''import google.generativeai as genai
from dotenv import load_dotenv
import os
from langchain_google_genai import GoogleGenerativeAIEmbeddings

load_dotenv("../dev.properties")
gemini_key = os.getenv("GEMINI_API_KEY")
print(gemini_key)

genai.configure(api_key=gemini_key)

for model in genai.list_models():
    if "embed" in model.name.lower():
        print(f"{model.name} → {model.supported_generation_methods}")'''


'import google.generativeai as genai\nfrom dotenv import load_dotenv\nimport os\nfrom langchain_google_genai import GoogleGenerativeAIEmbeddings\n\nload_dotenv("../dev.properties")\ngemini_key = os.getenv("GEMINI_API_KEY")\nprint(gemini_key)\n\ngenai.configure(api_key=gemini_key)\n\nfor model in genai.list_models():\n    if "embed" in model.name.lower():\n        print(f"{model.name} → {model.supported_generation_methods}")'

In [40]:
#Vector Store

class VectorStore:
    def __init__(self, collection_name:str = 'bhagavad_verses', persistent_directory:str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persistent_directory = persistent_directory
        self.client = None
        self.collection = None
        self._load_vector_store()

    def _load_vector_store(self):
        """initialze ChromaDB CLient and Collection"""
        try:
            print(f"Loading the Vector Store: {self.collection_name}")
            self.client = chromadb.PersistentClient(path=self.persistent_directory)
            self.collection = self.client.get_or_create_collection(name=self.collection_name,
            metadata={"description":"Bhagavad Gita Verses embeddings for RAG"})
            print(f"Vector Store Loaded Successfully! Collection: {self.collection.name}")
            print(f"Existing Collection Count: {self.collection.count()}")
        except Exception as e:
            print(f"Error loading vector store: {e}")
            raise e

    def add_documents(self, documents: List[Any], embeddings : np.ndarray):
        """Add documents and the corresponding embeddings to the vector store
        Args:
            documents : List of documents to add
            embeddings : List of embeddings corresponding to the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must be equal.")
        
        print(f"Adding {len(documents)} documents to the vector store...")
        
        #Prepare documents for ChromaDB
        ids =[]
        metadatas =[]
        documents_text = []
        embeddings_list = []

        for i, (doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id = f"doc_{hashlib.md5(doc.page_content.encode()).hexdigest()[:12]}"
            ids.append(doc_id)

            #Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            #Document Content
            documents_text.append(doc.page_content)

            #Embedding
            embeddings_list.append(embedding.tolist())

            #Add to Collection

        try:
            self.collection.upsert(
                documents=documents_text, 
                embeddings=embeddings_list, 
                metadatas=metadatas, 
                ids=ids)
            print(f"Documents added {len(documents)} documents successfully! Total Documents in Collection Count: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents: {e}")
            raise e

vectorstore = VectorStore()
vectorstore


    

Loading the Vector Store: bhagavad_verses
Vector Store Loaded Successfully! Collection: bhagavad_verses
Existing Collection Count: 640


In [51]:
#chunks

In [41]:
##Convert text to embeddings

texts = [doc.page_content for doc in chunks]
#texts

# Generate Embeddings for each chunk

#embeddings = embedding_manager.generateEmbeddings(texts)

#Store in Vector database

#vectorstore.add_documents(chunks,embeddings=embeddings)

print("Embeddings stored in ChromaDB")

Embeddings stored in ChromaDB


In [42]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage
import re
    

In [43]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store with LLM re-ranking"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager, api_key: str = gemini_key):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
            api_key: Gemini API key for LLM re-ranking
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
        self.reranker_llm = ChatGoogleGenerativeAI(
            model="gemini-2.5-flash",
            google_api_key=api_key,
            temperature=0.0,
        )
        print(f"📚 RAG Retriever ready — {self.vector_store.collection.count()} docs")
    def retrieve(self, query: str, top_k: int = 5, where_filter: dict = None, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            where_filter: Optional metadata filter, e.g. {"chapter_title": "Karm Yog"}
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding (embed_query — optimized for search)
        query_embedding = self.embedding_manager.model.embed_query(query)

        #print(f"Embedded Query: {query_embedding}")
        
        # Search in vector store
        try:
            query_params = {
                "query_embeddings": [query_embedding],
                "n_results": top_k,
                "include": ["documents", "metadatas", "distances"],
            }
            if where_filter:
                query_params["where"] = where_filter
            results = self.vector_store.collection.query(**query_params)
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = round(1 - distance, 4)
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        
    def llm_rerank(self, query: str, results: List[Dict[str, Any]], final_k: int = 5) -> List[Dict[str, Any]]:
        """
        Use Gemini Flash to re-rank retrieved results by relevance
        
        Args:
            query: The user's original question
            results: List of retrieved docs from self.retrieve()
            final_k: How many to keep after re-ranking
            
        Returns:
            Top final_k results, re-ordered by LLM judgment
        """
        if len(results) <= final_k:
            return results
        # Build candidates list for the LLM
        candidates = "\n\n".join([
            f"[{i+1}] (Verse: {r['metadata'].get('source', '?')} | {r['metadata'].get('chapter_title', '?')})\n{r['content'][:300]}"
            for i, r in enumerate(results)
        ])
        prompt = f"""You are a scripture relevance judge. Given the user's question, rank the following scripture passages from MOST to LEAST relevant.
        USER'S QUESTION: "{query}"
        PASSAGES:
        {candidates}
        Return ONLY the passage numbers in order of relevance, separated by commas.
        Example: 3, 1, 5, 2, 4
        Do not explain, just return the numbers."""
        try:
            response = self.reranker_llm.invoke([HumanMessage(content=prompt)])
            ranking_text = response.content.strip()
            
            # Parse "3, 1, 5, 2, 4" → [3, 1, 5, 2, 4]
            ranked_indices = [int(x.strip()) - 1 for x in re.findall(r'\d+', ranking_text)]
            
            # Reorder results based on LLM ranking
            reranked = []
            for idx in ranked_indices:
                if 0 <= idx < len(results):
                    results[idx]['rank'] = len(reranked) + 1
                    reranked.append(results[idx])
            
            print(f"📊 LLM re-ranked {len(results)} → top {final_k}")
            return reranked[:final_k]
        
        except Exception as e:
            print(f"⚠️ Re-ranking failed ({e}), returning original order")
            return results[:final_k]
    def retrieve_and_rerank(self, query: str, initial_k: int = 10, final_k: int = 5, 
                            where_filter: dict = None, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Full pipeline: Retrieve broadly → LLM re-rank → Return top results
        
        Args:
            query: User's question
            initial_k: How many to retrieve initially (wide net)
            final_k: How many to keep after re-ranking
            where_filter: Optional metadata filter
            score_threshold: Minimum similarity score
            
        Returns:
            Top final_k re-ranked results
        """
        # Step 1 — Cast wide net
        results = self.retrieve(query, top_k=initial_k, where_filter=where_filter, score_threshold=score_threshold)
        
        # Step 2 — LLM re-rank
        reranked = self.llm_rerank(query, results, final_k=final_k)
        
        return reranked
    def pretty_print(self, results: List[Dict[str, Any]]) -> None:
        """Display retrieval results in a readable format"""
        print("=" * 60)
        print("📖  RETRIEVED VERSES")
        print("=" * 60)
        for r in results:
            meta = r["metadata"]
            print(f"\n--- Rank {r['rank']} (similarity: {r['similarity_score']}) ---")
            print(f"  📌 {meta.get('chapter_title', '?')} | Verse: {meta.get('source', '?')}")
            print(f"  📝 {r['content'][:300]}")
        print("=" * 60)
rag_retriever = RAGRetriever(vectorstore, embedding_manager)

📚 RAG Retriever ready — 640 docs


In [44]:
rag_retriever

In [ ]:
#rag_retriever.retrieve("I lost someone close to me, how to cope with grief")
#rag_retriever.retrieve_and_rerank("I am not getting promotion, although I am working hard")


Retrieving documents for query: 'I am not getting promotion, although I am working hard'
Top K: 10, Score threshold: 0.0
Embedded Query: [-0.0024101208, 0.01931355, 0.021694655, -0.06040132, -0.002192746, 0.013863903, 0.0015502954, 0.023520464, -0.013580777, 0.0055837478, 0.0068803634, -0.041335117, 0.021603383, 0.024103133, 0.10339066, 0.028044876, 0.0088031925, 0.0020061224, 0.028973816, -0.006455971, -0.015807135, -0.0011757552, 0.025327107, -0.00025567587, -0.009992307, -0.03302743, -0.0013064009, 0.013321437, -0.004928539, -0.026532155, -0.005077988, 0.012262306, -0.009089998, 0.017119704, -0.01288249, 0.02074334, 0.012846024, -0.02179336, 0.014133326, -0.006737074, 0.005921308, 0.021357836, -0.021857712, 0.009411624, -0.008706218, -0.004732735, 0.0127221495, -0.022549352, -0.01968959, -0.015386485, -0.011949972, 0.0189035, -0.004423849, -0.16897863, -0.011820686, 0.0038295265, 0.015212496, 0.008589117, 0.008920785, -0.035177052, -0.013951224, 0.012621662, -0.009005752, 0.00676684

[{'id': 'doc_52306a87f96f',
  'content': 'chapter_number: Chapter 2\nchapter_verse: 2.47\ntranslation: You have a right to perform your prescribed duties, but you are not entitled to the fruits of your actions. Never consider yourself to be the cause of the results of your activities, nor be attached to inaction.',
  'metadata': {'content_length': 270,
   'source': '2.47',
   'row': 80,
   'doc_index': 80,
   'chapter_title': 'Sānkhya Yog'},
  'similarity_score': 0.2376,
  'distance': 0.7623911499977112,
  'rank': 1},
 {'id': 'doc_1d027563dcc8',
  'content': 'chapter_number: Chapter 3\nchapter_verse: 3.9\ntranslation: Work must be done as a yajna to the Supreme Lord; otherwise, work causes bondage in this material world. Therefore, O son of Kunti, for the satisfaction of God, perform your prescribed duties, without being attached to the results.',
  'metadata': {'row': 113,
   'content_length': 289,
   'chapter_title': 'Karm Yog',
   'source': '3.9',
   'doc_index': 113},
  'similarity

Integration of VectorDB Contexts along with LLM Out-put

In [58]:
## Advanced RAG Pipeline — Vedic Life Coach Generation Layer

class PromptBuilder:
    """Constructs system and user prompts for the LLM"""
    
    SYSTEM_PROMPT = """You are a wise, compassionate Vedic Life Coach deeply versed in ancient Indian scriptures — including the Bhagavad Gita, Upanishads, Vedas, and other sacred Hindu texts.
    INSTRUCTIONS:
    1. Read the RETRIEVED VERSES carefully — these are your ONLY source of truth.
    2. Understand the user's life situation with empathy.
    3. Connect the teachings from the verses to their specific situation in practical, modern terms.
    4. Always cite the exact source (scripture name, chapter, and verse number) when referencing a teaching.
    5. If verses from multiple scriptures are retrieved, weave their wisdom together into a unified, coherent response.
    6. End with ONE clear, actionable takeaway the user can apply today.
    RULES:
    - NEVER fabricate or invent scripture quotes. ONLY reference the provided verses.
    - If the verses don't directly address the question, use the closest teachings and say so honestly.
    - Be warm and encouraging — like a caring mentor, not a lecturer.
    - Respect the context of each scripture — do not misattribute teachings across texts.
    FORMATTING:
    - Keep your response under 500 words.
    - Use 2-3 short paragraphs maximum.
    - One or two key teaching, one practical takeaway. That's it.
    - No long lists or bullet points."""

    @staticmethod
    def build_context(results: List[Dict[str, Any]]) -> str:
        """Format retrieved verses into a context block for the LLM
        Args:
            results: List of retrieved documents from RAGRetriever
        Returns:
            Formatted string of verses for the LLM prompt
        """
        if not results:
            return "No relevant verses were found."
        
        verses = []
        for i, r in enumerate(results, 1):
            meta = r["metadata"]
            verse_ref = meta.get("source", "unknown")
            chapter = meta.get("chapter_title", "unknown")
            verses.append(
                f"[Verse {i}] {chapter} — Verse {verse_ref}\n"
                f"{r['content']}"
            )
        return "\n\n".join(verses)

    @staticmethod
    def build_messages(question: str, context: str) -> List:
        """Build the final message list for the LLM
        Args:
            question: User's life question
            context: Formatted scripture context
        Returns:
            List of SystemMessage and HumanMessage for LLM
        """
        user_prompt = (
            f"RETRIEVED SCRIPTURE VERSES:\n"
            f"{'━' * 50}\n"
            f"{context}\n"
            f"{'━' * 50}\n\n"
            f"USER'S LIFE SITUATION:\n"
            f"{question}\n\n"
            f"Provide compassionate guidance grounded in the above verses."
        )
        return [
            SystemMessage(content=PromptBuilder.SYSTEM_PROMPT),
            HumanMessage(content=user_prompt),
        ]


class AdvancedRAGPipeline:
    """
    End-to-end RAG pipeline for Vedic Life Coaching
    
    Flow: User Question → Retrieve Verses → Build Prompt → LLM → Cited Answer
    
    Components:
        - RAGRetriever:    Semantic search + LLM re-ranking
        - PromptBuilder:   System prompt + context formatting  
        - LLM (Gemini):    Generates scripture-grounded responses
    """
    
    def __init__(self, retriever: RAGRetriever, api_key: str = gemini_key,
                 model_name: str = "gemini-3.1-pro-preview", temperature: float = 0.6):
        """
        Initialize the Advanced RAG Pipeline
        
        Args:
            retriever:    RAGRetriever instance
            api_key:      Gemini API key
            model_name:   LLM model for response generation
            temperature:  Creativity control (0.0=focused, 1.0=creative)
        """
        self.retriever = retriever
        self.prompt_builder = PromptBuilder()
        self.llm = ChatGoogleGenerativeAI(
            model=model_name,
            google_api_key=api_key,
            temperature=temperature,
        )
        self.history: List[Dict[str, Any]] = []
        print(f"🙏 Vedic Life Coach ready (LLM: {model_name})")
    
    def query(self, question: str, top_k: int = 5, use_reranking: bool = True,
              where_filter: dict = None, min_score: float = 0.0,
              summarize: bool = False) -> Dict[str, Any]:
        """
        Run the full RAG pipeline on a user question
        
        Args:
            question:       The user's life question or situation
            top_k:          Number of verses to use in context
            use_reranking:  Whether to use LLM re-ranking
            where_filter:   Filter by metadata, e.g. {"chapter_title": "Karm Yog"}
            min_score:      Minimum similarity score threshold
            summarize:      Whether to generate a 2-sentence summary
            
        Returns:
            Dict with keys: question, answer, sources, citations, summary
        """
        # ── Step 1: Retrieve relevant verses ──────────────────
        print(f"\n🔍 Step 1: Retrieving verses for: '{question}'")
        if use_reranking:
            results = self.retriever.retrieve_and_rerank(
                question, initial_k=top_k * 2, final_k=top_k,
                where_filter=where_filter, score_threshold=min_score
            )
        else:
            results = self.retriever.retrieve(
                question, top_k=top_k,
                where_filter=where_filter, score_threshold=min_score
            )
        
        if not results:
            return {
                "question": question,
                "answer": "🙏 I couldn't find relevant verses. Please try rephrasing.",
                "sources": [], "citations": [], "summary": None,
            }
        
        # ── Step 2: Build prompt with context ─────────────────
        print(f"📝 Step 2: Building prompt with {len(results)} verses")
        context = self.prompt_builder.build_context(results)
        messages = self.prompt_builder.build_messages(question, context)
        
        # ── Step 3: Generate LLM response ─────────────────────
        print(f"🙏 Step 3: Generating response...")
        response = self.llm.invoke(messages)
        answer = response.content

        # Handle different response formats
        if isinstance(answer, list):
            # Extract text from each part
            parts = []
            for part in answer:
                if isinstance(part, dict) and 'text' in part:
                    parts.append(part['text'])
                elif isinstance(part, str):
                    parts.append(part)
                else:
                    parts.append(str(part))
            answer = "\n".join(parts)
        elif isinstance(answer, dict) and 'text' in answer:
            answer = answer['text']
        
        # ── Step 4: Build citations ───────────────────────────
        citations = []
        for i, r in enumerate(results, 1):
            meta = r["metadata"]
            citations.append(
                f"[{i}] {meta.get('chapter_title', '?')} — Verse {meta.get('source', '?')} "
                f"(relevance: {r['similarity_score']})"
            )
        
        answer_with_citations = answer + "\n\n📚 Scripture References:\n" + "\n".join(citations)
        
        # ── Step 5: Optional summary ──────────────────────────
        summary = None
        if summarize:
            print("📊 Step 5: Generating summary...")
            summary_resp = self.llm.invoke([
                HumanMessage(content=f"Summarize this guidance in exactly 2 sentences:\n\n{answer}")
            ])
            summary = summary_resp.content
        
        # ── Step 6: Store in history ──────────────────────────
        result = {
            "question": question,
            "answer": answer_with_citations,
            "sources": results,
            "citations": citations,
            "summary": summary,
        }
        self.history.append(result)
        print(f"✅ Done! ({len(self.history)} queries in history)\n")
        
        return result
    
    def display(self, result: Dict[str, Any]) -> None:
        """Pretty-print a query result"""
        print("=" * 60)
        print("🙏  VEDIC LIFE COACH")
        print("=" * 60)
        print(f"\n Your Question: {result['question']}\n")
        print("─" * 60)
        print(f"\n{result['answer']}")
        if result.get("summary"):
            print(f"\n📝 Summary: {result['summary']}")
        print("\n" + "=" * 60)
    
    def get_history(self) -> List[Dict[str, Any]]:
        """Return full conversation history"""
        return self.history

# Initialize
coach = AdvancedRAGPipeline(rag_retriever)



    

🙏 Vedic Life Coach ready (LLM: gemini-3.1-pro-preview)


In [60]:
result = coach.query("I easily get distracted and cannot focus on my work. What should I do?")
coach.display(result)



🔍 Step 1: Retrieving verses for: 'I easily get distracted and cannot focus on my work. What should I do?'
Retrieving documents for query: 'I easily get distracted and cannot focus on my work. What should I do?'
Top K: 10, Score threshold: 0.0
Retrieved 10 documents (after filtering)
📊 LLM re-ranked 10 → top 5
📝 Step 2: Building prompt with 5 verses
🙏 Step 3: Generating response...
✅ Done! (2 queries in history)

🙏  VEDIC LIFE COACH

 Your Question: I easily get distracted and cannot focus on my work. What should I do?

────────────────────────────────────────────────────────────

It is completely natural to feel overwhelmed by a wandering attention span; you are certainly not alone in this struggle. Even the great warrior Arjun felt this exact frustration. In the Bhagavad Gita (Chapter 6, Verse 35), Lord Krishna validates this deeply human experience, saying, "the mind is indeed very difficult to restrain." He warns that an unbridled mind makes any disciplined effort difficult to mast